# Telecom Customer Churn: Data Cleaning and Preparation

This notebook loads the Telco Customer Churn CSV, explores and cleans it, creates a few useful features, scales numeric columns, splits the data, and exports the prepared files.

**Dataset:** `Telco-Customer-Churn.csv`. On Colab keep it in `MyDrive/Data-Science`; on local Jupyter keep it next to this notebook. If the file is not found, the public IBM copy of the same dataset is downloaded.

In [ ]:
# Mount Google Drive (run in Google Colab)
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CSV_PATH = '/content/drive/MyDrive/Data-Science/Telco-Customer-Churn.csv'
except ImportError:  # local Jupyter
    CSV_PATH = 'Telco-Customer-Churn.csv'

if not os.path.exists(CSV_PATH):  # fall back to the public copy of the same dataset
    CSV_PATH = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from pathlib import Path

# Load data
raw = pd.read_csv(CSV_PATH)
print('Rows and columns:', raw.shape)
display(raw.head())

## 1. Explore the data

In [ ]:
display(raw.info())

display(raw.describe(include='all').T)
print('Missing values by column:')
display(raw.isna().sum().to_frame('missing_count'))
print('Duplicate rows:', raw.duplicated().sum())

## 2. Clean missing values and duplicates

Blank strings are treated as missing. Numeric columns are filled with their median; other columns use their most common value. Duplicate records are removed.

In [ ]:
df = raw.copy()
df.columns = df.columns.str.strip()
df = df.replace(r'^\s*$', np.nan, regex=True)
df = df.drop_duplicates().copy()

# Convert TotalCharges to numeric when that column is present (blank values become missing)
if 'TotalCharges' in df.columns:
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

for col in df.columns:
    if df[col].isna().any():
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(df[col].median())
        else:
            mode = df[col].mode(dropna=True)
            fill_value = mode.iloc[0] if not mode.empty else 'Unknown'
            df[col] = df[col].fillna(fill_value)
print('Shape after missing-value handling and deduplication:', df.shape)
print('Remaining missing values:', int(df.isna().sum().sum()))

## 3. Standardize values and data types

Trim whitespace from text fields and normalize common Yes/No and gender labels.

In [ ]:
for col in df.select_dtypes(include=['object','string']).columns:
    df[col] = df[col].astype('string').str.strip()

for col in df.select_dtypes(include=['object','string']).columns:
    lowered = df[col].str.lower()
    df.loc[lowered.isin(['yes','no']), col] = lowered[lowered.isin(['yes','no'])].str.capitalize()
    df.loc[lowered.isin(['male','female']), col] = lowered[lowered.isin(['male','female'])].str.capitalize()

if 'SeniorCitizen' in df.columns:
    df['SeniorCitizen'] = df['SeniorCitizen'].astype('int64')
if 'tenure' in df.columns:
    df['tenure'] = pd.to_numeric(df['tenure'], errors='coerce').fillna(0).astype('int64')
if 'TotalCharges' in df.columns:
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
display(df.dtypes.to_frame('data_type'))

## 4. Check outliers

Show the IQR outlier counts in numeric columns. For this customer dataset, flag them for review instead of automatically deleting valid high-value or long-tenure customers.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
outlier_counts = {}
for col in numeric_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    outlier_counts[col] = int(((df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)).sum())
pd.Series(outlier_counts, name='IQR outlier count').to_frame()

## 5. Create simple features

Create tenure groups and a monthly-charge-to-total-charge ratio when the required columns are present.

In [ ]:
if 'tenure' in df.columns:
    df['TenureGroup'] = pd.cut(df['tenure'], bins=[-1, 12, 24, 48, np.inf], labels=['0-12', '13-24', '25-48', '49+'])
if {'MonthlyCharges','TotalCharges'}.issubset(df.columns):
    df['MonthlyToTotalRatio'] = df['MonthlyCharges'] / df['TotalCharges'].replace(0, np.nan)
    df['MonthlyToTotalRatio'] = df['MonthlyToTotalRatio'].replace([np.inf, -np.inf], np.nan).fillna(0)
display(df.head())

## 6. Prepare features and split train/test

The target is `Churn`. Identifier columns are excluded from model features. Categorical features are one-hot encoded; numeric features are standardized. The scaler is fitted only on the training set to avoid data leakage.

In [ ]:
if 'Churn' not in df.columns:
    raise ValueError('Expected target column Churn was not found.')

id_cols = [c for c in ['customerID','CustomerID'] if c in df.columns]

X = df.drop(columns=['Churn'] + id_cols)
y = df['Churn'].astype(str).str.strip()

X = pd.get_dummies(X, drop_first=True, dtype=int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,
    stratify=y if y.nunique() > 1 else None
)

scale_cols = X_train.select_dtypes(include=np.number).columns
scaler = StandardScaler()
X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])

print('Train:', X_train.shape, 'Test:', X_test.shape)

## 7. Export the prepared data

In [ ]:
OUT_DIR = Path('telco_prepared')
OUT_DIR.mkdir(exist_ok=True)
df.to_csv(OUT_DIR / 'telco_cleaned.csv', index=False)
X_train.to_csv(OUT_DIR / 'X_train.csv', index=False)
X_test.to_csv(OUT_DIR / 'X_test.csv', index=False)
y_train.to_csv(OUT_DIR / 'y_train.csv', index=False)
y_test.to_csv(OUT_DIR / 'y_test.csv', index=False)
print('Saved prepared files to:', OUT_DIR.resolve())